# PACE Phase 8 — independent pretrained ESM-2 benchmark

**Analysis only; no manuscript or supplementary files are edited.**

Compares the original **90 physicochemical/CTD descriptors** with the **pretrained ESM-2 (6-layer, 8M, 320D)** protein-language-model embeddings, and a combined representation. Same cleaned benchmark datasets and same train/test splits are used for all methods.

- Classification: AMPlify 6,271 train / 1,568 held-out, with amPEPpy 1,848 leakage-filtered external peptides.
- Quantitative: six GRAMPA ESKAPE endpoints and 1,271 hemolysis peptides (five-fold CV, same folds within each endpoint).
- All source SHA256 checksums, model revision ID, embeddings, predictions, and metrics are exported for audit.

**Prerequisites:** Run this notebook in Google Colab with a GPU runtime if possible. It downloads public GitHub source data and official pretrained ESM-2 model weights. Do not use random/untrained models. This does *not* constitute laboratory validation, nor does it change PACE original results.

In [ ]:
%pip -q install "transformers==4.46.3" "huggingface_hub>=0.26,<1" "biopython>=1.80" "scikit-learn>=1.4,<2" pandas numpy requests safetensors

In [ ]:
import os, sys, io, json, time, random, hashlib, zipfile, platform
from pathlib import Path
import requests, pandas as pd, numpy as np, torch, sklearn
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score,
                             average_precision_score, r2_score, mean_absolute_error,
                             mean_squared_error)
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModel
SEED=2024
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
O=Path('/content/PACE_Phase8_ESM2_Output');O.mkdir(parents=True,exist_ok=True)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
MODEL='facebook/esm2_t6_8M_UR50D'
print('Device',DEVICE,'Torch',torch.__version__,'Sklearn',sklearn.__version__)

In [ ]:
# Download EXACT source filenames, archive SHA256, and apply original curation rules.
root_am='https://raw.githubusercontent.com/BirolLab/AMPlify/master/data/'
root_pp='https://raw.githubusercontent.com/tlawrence3/amPEPpy/master/training_data/'
root_gr='https://raw.githubusercontent.com/zswitten/Antimicrobial-Peptides/master/'
urls={
 'AMP_train':root_am+'AMPlify_AMP_train_common.fa',
 'AMP_test':root_am+'AMPlify_AMP_test_common.fa',
 'nonAMP_train':root_am+'AMPlify_non_AMP_train_balanced.fa',
 'nonAMP_test':root_am+'AMPlify_non_AMP_test_balanced.fa',
 'external_AMP':root_pp+'M_model_train_AMP_sequence.numbered.fasta',
 'external_nonAMP':root_pp+'M_model_train_nonAMP_sequence.numbered.proplen.subsample.fasta',
 'GRAMPA':root_gr+'data/grampa.csv',
 'Hemolysis':root_gr+'Hemolysis/Cleaned_hemolytic_data.csv',
}
session=requests.Session();binary={}; provenance={}
for key,url in urls.items():
  resp=session.get(url,timeout=120);resp.raise_for_status()
  binary[key]=resp.content
  provenance[key]={'url':url,'sha256':hashlib.sha256(resp.content).hexdigest(),'bytes':len(resp.content)}
  print(key,len(resp.content),'bytes')

def readfa(data):
  out=[];buf=[];active=False
  for line in data.decode('utf-8').splitlines():
    line=line.strip()
    if line.startswith('>'):
      if active:out.append(''.join(buf))
      active=True;buf=[]
    elif active:buf.append(line)
  if active:out.append(''.join(buf))
  return out

In [ ]:
# Exact original user-supplied PACE descriptor engine is embedded for provenance.
from pathlib import Path
features_path=Path('/content/pace_original_features.py')
features_path.write_text('"""Peptide descriptor engine (PACE-70 feature set).\n\nAll scales are literature-standard and hard-coded here so the pipeline is fully\nreproducible offline. No values are invented: sources are named in each block.\n"""\nimport numpy as np\n\nAA = list("ACDEFGHIKLMNPQRSTVWY")\nAAset = set(AA)\n\n# Kyte & Doolittle (1982) J Mol Biol 157:105-132 hydropathy\nKD = dict(A=1.8, R=-4.5, N=-3.5, D=-3.5, C=2.5, Q=-3.5, E=-3.5, G=-0.4, H=-3.2,\n          I=4.5, L=3.8, K=-3.9, M=1.9, F=2.8, P=-1.6, S=-0.8, T=-0.7, W=-0.9,\n          Y=-1.3, V=4.2)\n\n# Eisenberg et al. (1984) consensus normalised hydrophobicity\nEIS = dict(A=0.62, R=-2.53, N=-0.78, D=-0.90, C=0.29, Q=-0.85, E=-0.74, G=0.48,\n           H=-0.40, I=1.38, L=1.06, K=-1.50, M=0.64, F=1.19, P=0.12, S=-0.18,\n           T=-0.05, W=0.81, Y=0.26, V=1.08)\n\n# Boman (2003) J Intern Med 254:197-215 protein-binding potential (kcal/mol)\nBOMAN = dict(L=-4.92, I=-4.92, V=-4.04, F=-2.98, M=-2.35, W=-2.33, A=-1.81,\n             C=-1.28, G=-0.94, Y=0.14, T=2.57, S=3.40, H=4.66, Q=5.54, K=5.55,\n             N=6.64, E=6.81, D=8.72, R=14.92, P=0.0)\n\n# pKa values (EMBOSS set, as used by Bio.SeqUtils.IsoelectricPoint)\nPKA_SIDE = dict(D=3.65, E=4.25, C=8.18, Y=10.07, H=6.00, K=10.53, R=12.48)\nPK_NTERM, PK_CTERM = 8.6, 3.6\n\n# Flexibility (Vihinen et al. 1994) average flexibility index\nFLEX = dict(A=0.360, C=0.350, D=0.510, E=0.500, F=0.310, G=0.540, H=0.320,\n            I=0.460, K=0.470, L=0.370, M=0.300, N=0.460, P=0.510, Q=0.490,\n            R=0.530, S=0.510, T=0.440, V=0.390, W=0.310, Y=0.420)\n\nGROUPS = {\n    "tiny": set("ACGST"), "small": set("ABCDGNPSTV"), "aliphatic": set("AILV"),\n    "aromatic": set("FHWY"), "nonpolar": set("ACFGILMPVWY"), "polar": set("DEHKNQRSTZ"),\n    "charged": set("DEHKR"), "basic": set("HKR"), "acidic": set("DE"),\n    "cysteine": set("C"), "proline": set("P"), "glycine": set("G"),\n}\n\n# CTD attribute groupings (Dubchak et al. 1995; Govindan & Nair 2011)\nCTD_ATTR = {\n    "hydrophobicity": ("RKEDQN", "GASTPHY", "CLVIMFW"),\n    "vdw_volume":     ("GASTPDC", "NVEQIL", "MHKFRYW"),\n    "polarity":       ("LIFWCMVY", "PATGS", "HQRKNED"),\n    "polarizability": ("GASDT", "CPNVEQIL", "KMHFRYW"),\n    "charge":         ("KR", "ANCQGHILMFPSTWYV", "DE"),\n    "ss_propensity":  ("EALMQKRH", "VIYCWFT", "GNPSD"),\n    "sa_accessibility": ("ALFCGIVW", "RKQEND", "MPSTHY"),\n}\n\nGLOBAL_NAMES = [\n    "length", "net_charge_pH7", "charge_density", "pI", "gravy_kd",\n    "hydrophobicity_eis", "hydrophobic_moment", "amphipathicity_ratio",\n    "aliphatic_index", "boman_index", "instability_like", "flexibility",\n    "frac_tiny", "frac_small", "frac_aliphatic", "frac_aromatic", "frac_nonpolar",\n    "frac_polar", "frac_charged", "frac_basic", "frac_acidic", "frac_cys",\n    "frac_pro", "frac_gly", "max_hydrophobic_run", "max_cationic_run",\n    "kr_ratio", "hydromoment_x_charge",\n]\n\n\ndef clean(seq):\n    s = "".join(ch for ch in str(seq).upper() if ch.isalpha())\n    return s if s and all(c in AAset for c in s) else None\n\n\ndef net_charge(seq, pH=7.4):\n    q = 1.0 / (1.0 + 10 ** (pH - PK_NTERM)) - 1.0 / (1.0 + 10 ** (PK_CTERM - pH))\n    for aa, pk in PKA_SIDE.items():\n        n = seq.count(aa)\n        if not n:\n            continue\n        if aa in "KRH":\n            q += n / (1.0 + 10 ** (pH - pk))\n        else:\n            q -= n / (1.0 + 10 ** (pk - pH))\n    return q\n\n\ndef isoelectric_point(seq):\n    """Bio.SeqUtils.IsoelectricPoint (EMBOSS pKa set) — reference implementation."""\n    from Bio.SeqUtils.IsoelectricPoint import IsoelectricPoint as IP\n    return float(IP(seq).pi())\n\n\ndef hydrophobic_moment(seq, angle=100, window=11):\n    """Eisenberg mean hydrophobic moment, max over sliding windows."""\n    n = len(seq)\n    w = min(window, n)\n    best = 0.0\n    rad = np.deg2rad(angle)\n    for i in range(n - w + 1):\n        sub = seq[i:i + w]\n        h = np.array([EIS[c] for c in sub])\n        idx = np.arange(w)\n        mu = np.sqrt((h * np.sin(idx * rad)).sum() ** 2 + (h * np.cos(idx * rad)).sum() ** 2)\n        best = max(best, mu / w)\n    return best\n\n\ndef aliphatic_index(seq):\n    n = len(seq)\n    return 100.0 * (seq.count("A") / n + 2.9 * seq.count("V") / n +\n                    3.9 * (seq.count("I") + seq.count("L")) / n)\n\n\ndef instability_like(seq):\n    """Dipeptide-instability proxy: mean Boman difference across adjacent pairs.\n    Reported as a relative index, not the Guruprasad II value."""\n    if len(seq) < 2:\n        return 0.0\n    return float(np.mean([abs(BOMAN[a] - BOMAN[b]) for a, b in zip(seq, seq[1:])]))\n\n\ndef max_run(seq, members):\n    best = cur = 0\n    for c in seq:\n        cur = cur + 1 if c in members else 0\n        best = max(best, cur)\n    return best\n\n\ndef global_features(seq):\n    n = len(seq)\n    q = net_charge(seq)\n    mu = hydrophobic_moment(seq)\n    h = float(np.mean([EIS[c] for c in seq]))\n    f = {g: sum(seq.count(c) for c in mem) / n for g, mem in GROUPS.items()}\n    return [\n        n, q, q / n, isoelectric_point(seq),\n        float(np.mean([KD[c] for c in seq])), h, mu,\n        mu / (abs(h) + 1e-6),\n        aliphatic_index(seq),\n        float(np.mean([BOMAN[c] for c in seq])),\n        instability_like(seq),\n        float(np.mean([FLEX[c] for c in seq])),\n        f["tiny"], f["small"], f["aliphatic"], f["aromatic"], f["nonpolar"],\n        f["polar"], f["charged"], f["basic"], f["acidic"], f["cysteine"],\n        f["proline"], f["glycine"],\n        max_run(seq, set("AILVFWMC")), max_run(seq, set("KR")),\n        (seq.count("K") + 1) / (seq.count("R") + 1),\n        mu * q,\n    ]\n\n\ndef aac(seq):\n    n = len(seq)\n    return [seq.count(a) / n for a in AA]\n\n\ndef ctd_ct(seq):\n    """CTD composition (3) + transition (3) per attribute."""\n    out = []\n    for attr, groups in CTD_ATTR.items():\n        code = []\n        for c in seq:\n            for gi, g in enumerate(groups):\n                if c in g:\n                    code.append(gi)\n                    break\n            else:\n                code.append(1)\n        code = np.array(code)\n        n = len(code)\n        out += [float((code == g).sum()) / n for g in range(3)]\n        if n > 1:\n            pairs = list(zip(code[:-1], code[1:]))\n            tot = len(pairs)\n            for a, b in [(0, 1), (0, 2), (1, 2)]:\n                out.append(sum(1 for x, y in pairs if {x, y} == {a, b}) / tot)\n        else:\n            out += [0.0, 0.0, 0.0]\n    return out\n\n\nCTD_NAMES = []\nfor _a in CTD_ATTR:\n    CTD_NAMES += [f"{_a}_C{g}" for g in (1, 2, 3)]\n    CTD_NAMES += [f"{_a}_T{t}" for t in ("12", "13", "23")]\n\nFEATURE_NAMES = GLOBAL_NAMES + [f"AAC_{a}" for a in AA] + CTD_NAMES\n\n\ndef featurize(seq):\n    return global_features(seq) + aac(seq) + ctd_ct(seq)\n\n\ndef featurize_many(seqs):\n    return np.asarray([featurize(s) for s in seqs], dtype=float)\n\n\nif __name__ == "__main__":\n    s = "GIGKFLHSAKKFGKAFVGEIMNS"  # magainin-2\n    v = featurize(s)\n    print("n_features =", len(v), "== len(names)", len(FEATURE_NAMES))\n    for k in ["length", "net_charge_pH7", "pI", "gravy_kd", "hydrophobic_moment"]:\n        print(f"  {k:22s} {v[FEATURE_NAMES.index(k)]:.3f}")\n')
import importlib.util
spec=importlib.util.spec_from_file_location('pace_original_features',str(features_path))
pacef=importlib.util.module_from_spec(spec);spec.loader.exec_module(pacef)
assert len(pacef.FEATURE_NAMES)==90
print('Original feature engine SHA256', hashlib.sha256(features_path.read_bytes()).hexdigest())

In [ ]:
def prep(data):
  seqs=[pacef.clean(s) for s in readfa(data)]
  return [s for s in seqs if s and 5<=len(s)<=60]
def labeled(positive,negative):
  a=prep(binary[positive]); b=prep(binary[negative])
  return pd.DataFrame({'sequence':a+b,'label':[1]*len(a)+[0]*len(b)}).drop_duplicates('sequence')
train=labeled('AMP_train','nonAMP_train')
test=labeled('AMP_test','nonAMP_test')
test=test.loc[~test.sequence.isin(set(train.sequence))].copy()
external=labeled('external_AMP','external_nonAMP')
external=external.loc[~external.sequence.isin(set(train.sequence)|set(test.sequence))].copy()
assert (len(train),int(train.label.sum()))==(6271,3124)
assert (len(test),int(test.label.sum()))==(1568,796)
assert (len(external),int(external.label.sum()))==(1848,671)
print('Classification: train',len(train),'test',len(test),'external',len(external))

rawg=pd.read_csv(io.BytesIO(binary['GRAMPA']))
g=rawg.loc[~rawg.has_unusual_modification.astype(bool)].copy()
g['seq']=g.sequence.map(pacef.clean)
g=g.dropna(subset=['seq']);g=g.loc[g.seq.str.len().between(5,60)].copy()
assert len(rawg)==51345 and len(g)==44151,(len(rawg),len(g))
eskape={'E. faecium':['E. faecium','E. faecalis'],
        'S. aureus':['S. aureus'],
        'K. pneumoniae':['K. pneumoniae'],
        'A. baumannii':['A. baumannii'],
        'P. aeruginosa':['P. aeruginosa'],
        'E. cloacae':['E. cloacae','E. aerogenes']}
quant={}
for name,species in eskape.items():
  x=g[g.bacterium.isin(species)].groupby('seq').value.median().reset_index()
  quant[name]=x.rename(columns={'seq':'sequence','value':'observed_log10'})
hemoraw=pd.read_csv(io.BytesIO(binary['Hemolysis']))
hemoraw['sequence']=hemoraw.Sequence.map(pacef.clean)
h=hemoraw.dropna(subset=['sequence']);h=h[h.sequence.str.len().between(5,60)]
quant['HC50']=h.groupby('sequence').log10_HC50.median().reset_index().rename(columns={'log10_HC50':'observed_log10'})
expected=[780,4128,743,306,2517,304,1271]
assert [len(d) for d in quant.values()]==expected,[len(d) for d in quant.values()]
print({k:len(v) for k,v in quant.items()})

In [ ]:
# Once per unique peptide, compute 90 original descriptors and pretrained ESM-2 embeddings.
union=sorted(set(pd.concat([train.sequence,test.sequence,external.sequence,
                *[d.sequence for d in quant.values()]],ignore_index=True)))
seq_index={s:i for i,s in enumerate(union)}
print('Unique sequences in all tasks:',len(union))
X90=np.asarray(pacef.featurize_many(union),dtype=np.float32)
assert X90.shape==(len(union),90)
# ESM model revision SHA is frozen for this run and recorded in final manifest.
model_revision=HfApi().model_info(MODEL).sha
print('ESM-2 model revision:',model_revision)
tok=AutoTokenizer.from_pretrained(MODEL,revision=model_revision)
model=AutoModel.from_pretrained(MODEL,revision=model_revision).to(DEVICE).eval()
emb=[];batch_size=48 if DEVICE=='cuda' else 8
with torch.inference_mode():
 for start in range(0,len(union),batch_size):
  batch=union[start:start+batch_size]
  tokens=tok(batch,return_tensors='pt',padding=True,truncation=False,
             return_special_tokens_mask=True)
  special=tokens.pop('special_tokens_mask').to(DEVICE)
  tokens={k:v.to(DEVICE) for k,v in tokens.items()}
  z=model(**tokens).last_hidden_state
  mask=(tokens['attention_mask'].bool() & ~special.bool()).unsqueeze(-1)
  pooled=(z*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1)
  emb.append(pooled.cpu().numpy().astype(np.float32))
  if start%2000<batch_size:print('Embedded',min(start+batch_size,len(union)),'/',len(union))
Xesm=np.concatenate(emb,axis=0);del model
if DEVICE=='cuda': torch.cuda.empty_cache()
assert Xesm.shape==(len(union),320) and np.isfinite(Xesm).all()
np.savez_compressed(O/'all_embeddings_and_features.npz',
                    sequences=np.array(union,dtype=str),X90=X90,ESM2_320D=Xesm)

In [ ]:
# Held-out and external classification comparison. SAME train/test sequences, SAME estimator per representation.
X21=X90[:,[0]+list(range(28,48))]
reprs={'21D_length_AAC':X21,'90D_PACE':X90,'ESM2_320D':Xesm,
       'Hybrid_90D_ESM2':np.concatenate([X90,Xesm],axis=1)}
tr=np.array([seq_index[s] for s in train.sequence]);te=np.array([seq_index[s] for s in test.sequence]);ex=np.array([seq_index[s] for s in external.sequence])
models={'Logistic':lambda:make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=3000,random_state=SEED)),
        'ExtraTrees200':lambda:ExtraTreesClassifier(n_estimators=200,n_jobs=-1,random_state=SEED)}
class_rows=[];class_preds=[]
for model_name,make in models.items():
 for repr_name,x in reprs.items():
  clf=make();clf.fit(x[tr],train.label.to_numpy())
  for split,indices,ds in [('heldout',te,test),('external',ex,external)]:
   pr=clf.predict_proba(x[indices])[:,1];yy=ds.label.to_numpy();yh=(pr>=0.5).astype(int)
   row={'model':model_name,'representation':repr_name,'split':split,'n':len(yy),
        'ROC_AUC':roc_auc_score(yy,pr),'PR_AUC':average_precision_score(yy,pr),
        'accuracy':accuracy_score(yy,yh),'MCC':matthews_corrcoef(yy,yh)}
   class_rows.append(row)
   class_preds.append(pd.DataFrame({'model':model_name,'representation':repr_name,
         'split':split,'sequence':ds.sequence.to_numpy(),'label':yy,'score':pr}))
   print(model_name,repr_name,split,'AUC',round(row['ROC_AUC'],4))
pd.DataFrame(class_rows).to_csv(O/'classification_metrics.csv',index=False)
pd.concat(class_preds,ignore_index=True).to_csv(O/'classification_predictions.csv',index=False)

In [ ]:
# Matched five-fold quantitative evaluation. Fixed 80-tree ExtraTrees on 90D, ESM-2 and hybrid.
# The held-out folds are defined by sequence at each endpoint, identical across representations.
quant_rows=[];quant_preds=[]
for endpoint,df in quant.items():
  ind=np.array([seq_index[s] for s in df.sequence]); y=df.observed_log10.to_numpy()
  folds=list(KFold(n_splits=5,shuffle=True,random_state=SEED).split(ind))
  for repname in ('90D_PACE','ESM2_320D','Hybrid_90D_ESM2'):
    x=reprs[repname][ind]; predictions=np.empty(len(y)); foldnum=np.empty(len(y),dtype=int)
    for j,(itr,ite) in enumerate(folds):
      clf=ExtraTreesRegressor(n_estimators=80,n_jobs=-1,random_state=SEED)
      clf.fit(x[itr],y[itr]);predictions[ite]=clf.predict(x[ite]);foldnum[ite]=j
    row={'endpoint':endpoint,'representation':repname,'n':len(y),
         'R2':r2_score(y,predictions),'RMSE':np.sqrt(mean_squared_error(y,predictions)),
         'MAE':mean_absolute_error(y,predictions)}
    quant_rows.append(row);print(endpoint,repname,'R2',round(row['R2'],4))
    quant_preds.append(pd.DataFrame({'endpoint':endpoint,'representation':repname,
           'sequence':df.sequence.to_numpy(),'fold':foldnum,'observed_log10':y,
           'predicted_log10':predictions}))
pd.DataFrame(quant_rows).to_csv(O/'quantitative_metrics.csv',index=False)
pd.concat(quant_preds,ignore_index=True).to_csv(O/'quantitative_predictions.csv',index=False)

In [ ]:
# Deliverable: compact audit ZIP for independent reanalysis.
manifest={'model':MODEL,'model_revision':model_revision,'seed':SEED,'device':DEVICE,
          'source_files':provenance,'n_train':len(train),'n_heldout':len(test),
          'n_external':len(external),'n_unique_sequences':len(union),
          'quantitative_sample_sizes':{k:len(v) for k,v in quant.items()},
          'input_feature_sha256':hashlib.sha256(features_path.read_bytes()).hexdigest(),
          'environment':{'python':platform.python_version(),'torch':torch.__version__,
                         'sklearn':sklearn.__version__}}
(O/'manifest.json').write_text(json.dumps(manifest,indent=2))
zippath=Path('/content/PACE_Phase8_ESM2_results.zip')
with zipfile.ZipFile(zippath,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in O.glob('*'): z.write(p,arcname=p.name)
 z.write(features_path,arcname='pace_original_features.py')
print('Finished. Results ZIP:',zippath,'size MB',round(zippath.stat().st_size/1e6,2))
try:
 from google.colab import files
 files.download(str(zippath))
except ImportError:
 print('Not in Colab; download ZIP from output directory.')